# 🧠 Aula 13 — Implementação de um Modelo Paralelo Simples (Síntese do Bloco 2)

**Curso:** Introdução a Arquitetura de Computadores — Senac  
**Bloco 2:** Programação, Otimização e Computação Heterogênea  

---

## 🎯 Objetivo da Aula
Desenvolver e comparar implementações paralelas (**GPU — CUDA Numba e CuPy**) e sequenciais (**CPU — Python puro e NumPy**) da **soma vetorial** e do **produto escalar**, gerando gráficos de speedup, um mini-relatório técnico e resolvendo **20 exercícios práticos e teóricos** para consolidar os aprendizados do Bloco 2.

### ⚙️ PASSO CRÍTICO: Ativar a GPU no Google Colab
1. Clique no menu superior **Ambiente de execução** (*Runtime*) ➔ **Alterar tipo de ambiente de execução** (*Change runtime type*).
2. Em **Acelerador de hardware**, escolha **T4 GPU** (ou equivalente NVIDIA).
3. Clique em **Salvar**.


## 1. Contextualização Teórica: Por que a GPU ganha? (SIMD vs. SIMT)

- **CPU — SIMD (Single Instruction, Multiple Data) / MIMD:** 8–16 núcleos potentes com execução sequencial por padrão ou vetorial curta (AVX-512).
- **GPU — SIMT (Single Instruction, Multiple Threads):** Milhares de threads em paralelo organizadas em Warps (32 threads) executando a mesma instrução sobre uma grade multidimensional.

Na soma vetorial $c[i] = a[i] + b[i]$ para $N = 10.000.000$:
- **CPU:** 10M operações efetuadas em série ou em pequenos blocos de núcleos.
- **GPU:** 10M operações distribuídas em dezenas de milhares de threads simultâneas em batches massivos.


## 2. Versão 1 & 2 — CPU: Python Puro e NumPy

Abaixo definimos as implementações de referência em CPU:
1. **Python Puro:** Laços nativos `for` (lento, sobrecarregado pelo interpretador).
2. **NumPy:** Vetorizado e utilizando rotinas C/BLAS altamente otimizadas para CPU.


In [ ]:
# @title 🐍 Implementações CPU (Python Puro & NumPy)
# ============================================================================
# OBJETIVO: Definir as versões de referência (baseline) que rodam na CPU.
# Elas servem de comparação para medir o ganho (speedup) obtido com a GPU.
# ============================================================================
import time  # time.perf_counter() = cronômetro de alta precisão (mede em segundos)
import numpy as np  # NumPy = vetorização e rotinas C/BLAS otimizadas para CPU


# ── Versão 1: Python Puro ──────────────────────────────────────────────────
# A forma mais lenta: o laço 'for' é interpretado pelo Python a cada iteração,
# adicionando um custo de interpretação para CADA elemento do vetor.
def soma_vetores_python(a, b):
    """Soma dois vetores elemento a elemento — laço Python puro."""
    resultado = [0.0] * len(a)  # Pré-aloca a lista de saída com o mesmo tamanho de 'a'
    for i in range(len(a)):     # Percorre cada índice individualmente
        resultado[i] = a[i] + b[i]  # Soma posição por posição
    return resultado

def multiplicacao_vetorial_python(a, b):
    """Produto escalar — redução sequencial em Python puro."""
    total = 0.0               # Acumulador do produto escalar
    for i in range(len(a)):   # Percorre todos os elementos
        total += a[i] * b[i]  # Multiplica e acumula (a[0]*b[0] + a[1]*b[1] + ...)
    return total


# ── Versão 2: CPU NumPy ────────────────────────────────────────────────────
# NumPy executa as operações em código C compilado (sem laço Python por elemento),
# o que já traz um enorme ganho sobre o Python puro.
def benchmark_numpy(N, repeticoes=50):
    """Mede o tempo médio de soma vetorial e produto escalar usando NumPy."""
    # Cria os vetores de entrada:
    # a = [1, 1, 1, ...] e b = [0, 1, 2, 3, ...], ambos em float32 (32 bits = 4 bytes)
    a = np.ones(N, dtype=np.float32)
    b = np.arange(N, dtype=np.float32)

    # Warm-up: executa uma vez fora do cronômetro para 'aquecer' caches/CPU
    # e evitar que a primeira execução (mais lenta) distorça a média.
    _ = a + b
    _ = np.dot(a, b)

    # Benchmark da soma vetorial: repete 'repeticoes' vezes e divide o tempo total
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        c = a + b  # Soma vetorizada (c[i] = a[i] + b[i])
    t_soma = (time.perf_counter() - t0) / repeticoes  # Tempo médio por execução

    # Benchmark do produto escalar (np.dot usa BLAS, extremamente otimizado)
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        s = np.dot(a, b)  # Produto escalar: soma(a[i] * b[i])
    t_dot = (time.perf_counter() - t0) / repeticoes

    return t_soma, t_dot  # Retorna os dois tempos médios (em segundos)


print("✅ Funções CPU (Python puro e NumPy) definidas com sucesso.")


## 3. Versão 3 — GPU: Kernels CUDA com Numba

Escrevemos kernels CUDA explícitos com Numba:
- **Kernel Soma:** Mapeia um thread para cada elemento do vetor (`cuda.grid(1)`).
- **Kernel Dot Product:** Realiza **redução paralela (Tree Reduction)** utilizando **Shared Memory** por bloco e atualização atômica no resultado final.


In [ ]:
# @title 🚀 Kernels CUDA com Numba (Soma Vetorial e Produto Escalar com Redução Paralela)
# ============================================================================
# OBJETIVO: Escrever kernels CUDA explícitos com Numba.
# Um 'kernel' é a função que roda NA GPU, executada por milhares de threads.
# Sintaxe de lançamento: kernel[numero_de_blocos, threads_por_bloco](argumentos)
# ============================================================================
import math
import numba
import numba.cuda as cuda


# ── Kernel CUDA: Soma Vetorial ─────────────────────────────────────────────
# @cuda.jit compila esta função Python para código CUDA executável na GPU.
@cuda.jit
def kernel_soma(a, b, c):
    """Cada thread soma um elemento."""
    # cuda.grid(1) = índice global da thread atual no grid 1D.
    # Ex.: bloco 2, thread 5 com 256 threads/bloco = 2*256 + 5 = 517.
    idx = cuda.grid(1)
    if idx < c.shape[0]:          # Guarda de limite: evita acessar fora do vetor
        c[idx] = a[idx] + b[idx]  # Cada thread calcula UM elemento da saída


# ── Kernel CUDA: Produto Escalar (Tree Reduction em Shared Memory) ────────
# O produto escalar gera UM único número, então precisamos REDUZIR (somar)
# todos os produtos parciais. Fazemos isso em duas etapas: dentro do bloco
# (árvore em shared memory) e entre os blocos (atomicAdd na memória global).
@cuda.jit
def kernel_dot_reducao(a, b, resultado_parcial):
    """
    Redução paralela em Shared Memory.
    Cada bloco soma parcialmente em árvore (Tree Reduction) e salva com atomicAdd.
    """
    # Memória compartilhada (SRAM) do bloco: rápida e visível a todas as threads do bloco.
    shared = cuda.shared.array(shape=256, dtype=numba.float32)
    tx  = cuda.threadIdx.x  # Índice LOCAL da thread dentro do bloco (0 a 255)
    idx = cuda.grid(1)      # Índice GLOBAL da thread no grid

    val = 0.0
    if idx < a.shape[0]:   # Cada thread calcula o produto a[i]*b[i] do seu elemento
        val = a[idx] * b[idx]
    shared[tx] = val       # Cada thread grava seu valor na posição local da shared memory
    cuda.syncthreads()     # BARREIRA: espera TODAS as threads do bloco terminarem a escrita

    # Árvore de redução dentro do bloco (soma aos pares):
    # Ex.: [a,b,c,d] -> [a+b, c+d] -> [(a+b)+(c+d)]  (log2(256)=8 passos)
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tx < stride:
            shared[tx] += shared[tx + stride]  # Soma a metade 'distante' à metade 'próxima'
        cuda.syncthreads()  # Garante que a soma do passo só comece após todas concluírem
        stride //= 2        # A cada passo, metade das threads ficam ativas

    # Thread 0 de cada bloco escreve o acumulado do bloco no resultado global.
    # atomic.add evita condição de corrida: vários blocos somando na MESMA posição.
    if tx == 0:
        cuda.atomic.add(resultado_parcial, 0, shared[0])


def benchmark_gpu_numba(N, repeticoes=50):
    """Transfere dados para a GPU e mede o tempo médio dos dois kernels."""
    # Vetores na memória do HOST (RAM), criados com NumPy:
    a_h = np.ones(N, dtype=np.float32)
    b_h = np.arange(N, dtype=np.float32)

    # Transferência Host -> Device (RAM -> VRAM) via barramento PCIe:
    a_d = cuda.to_device(a_h)  # Copia 'a' para a GPU
    b_d = cuda.to_device(b_h)  # Copia 'b' para a GPU
    c_d = cuda.device_array(N, dtype=np.float32)  # Aloca saída DIRETO na VRAM (sem copiar)

    # Configuração do grid: 256 threads por bloco e o número de blocos necessário
    # para cobrir os N elementos (divisão arredondada para cima).
    THREADS_POR_BLOCO = 256
    blocos = math.ceil(N / THREADS_POR_BLOCO)

    # Warm-up: primeira chamada compila o kernel e aquece a GPU (mais lenta).
    kernel_soma[blocos, THREADS_POR_BLOCO](a_d, b_d, c_d)
    res_d = cuda.to_device(np.zeros(1, dtype=np.float32))  # Acumulador do dot na GPU
    kernel_dot_reducao[blocos, THREADS_POR_BLOCO](a_d, b_d, res_d)
    cuda.synchronize()  # Bloqueia a CPU até a GPU terminar todo o trabalho enviado

    # Benchmark Soma: lança o kernel 'repeticoes' vezes e mede o tempo total
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        kernel_soma[blocos, THREADS_POR_BLOCO](a_d, b_d, c_d)
    cuda.synchronize()  # IMPORTANTE: sem sync, mediríamos só o tempo de ENFILEIRAR, não de rodar
    t_soma = (time.perf_counter() - t0) / repeticoes  # Tempo médio por lançamento

    # Benchmark Dot Product
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        res_d = cuda.to_device(np.zeros(1, dtype=np.float32))  # Zera o acumulador a cada iteração
        kernel_dot_reducao[blocos, THREADS_POR_BLOCO](a_d, b_d, res_d)
    cuda.synchronize()
    t_dot = (time.perf_counter() - t0) / repeticoes

    return t_soma, t_dot  # Tempos médios em segundos


print("✅ Kernels Numba CUDA configurados com sucesso.")


## 4. Versão 4 — GPU: CuPy (NumPy Acelerado na GPU)

CuPy provê uma API idêntica ao NumPy rodando diretamente no CUDA da GPU sem a necessidade de escrever kernels manuais.


In [ ]:
# @title ⚡ Versão CuPy (NumPy na GPU)
# ============================================================================
# OBJETIVO: Usar a API do CuPy — idêntica à do NumPy, mas executando na GPU.
# Não escrevemos kernels: o CuPy já traz operações otimizadas prontas.
# ============================================================================
import cupy as cp  # CuPy = 'NumPy da GPU'; cada operação vira um kernel CUDA


def benchmark_cupy(N, repeticoes=50):
    """Mede soma vetorial e produto escalar usando CuPy (tudo na VRAM)."""
    # Cria os vetores DIRETO na memória da GPU (não há cópia da RAM aqui).
    a_cp = cp.ones(N, dtype=cp.float32)     # Equivalente a np.ones, mas na VRAM
    b_cp = cp.arange(N, dtype=cp.float32)   # Equivalente a np.arange, mas na VRAM

    # Warm-up: primeira execução compila/aquece os kernels internos do CuPy.
    _ = a_cp + b_cp
    _ = cp.dot(a_cp, b_cp)
    cp.cuda.runtime.deviceSynchronize()  # Espera a GPU terminar antes de cronometrar

    # Benchmark Soma: as operações são assíncronas; o sync garante a medição real.
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        c_cp = a_cp + b_cp  # Soma vetorizada na GPU
    cp.cuda.runtime.deviceSynchronize()
    t_soma = (time.perf_counter() - t0) / repeticoes

    # Benchmark Dot Product
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        s_cp = cp.dot(a_cp, b_cp)  # Produto escalar na GPU
    cp.cuda.runtime.deviceSynchronize()
    t_dot = (time.perf_counter() - t0) / repeticoes

    return t_soma, t_dot


print("✅ CuPy configurado com sucesso.")


## 5. Coleta Completa de Benchmark & Varredura de $N$

Varremos $N \in [10.000, 100.000, 1.000.000, 10.000.000, 100.000.000]$ efetuando no mínimo 50 repetições com warm-up prévio.


In [ ]:
# @title 📊 Executar Benchmark Completo das 4 Versões
# ============================================================================
# OBJETIVO: Rodar todas as versões para vários tamanhos de vetor (N) e montar
# uma tabela comparativa (DataFrame) com tempos e speedups.
# ============================================================================
import pandas as pd  # Pandas organiza os resultados em uma tabela (DataFrame)

# Tamanhos de vetor testados (de 10 mil a 100 milhões de elementos)
Ns = [10_000, 100_000, 1_000_000, 10_000_000, 100_000_000]
repeticoes = 50  # Número de repetições para tirar a média de cada medição

registros = []  # Lista onde cada N gera um 'registro' (linha da tabela)

# Cabeçalho da tabela impressa no console
print(f"{'N':>12} | {'Soma NumPy':>12} | {'Soma CUDA':>12} | {'Soma CuPy':>12} | {'Dot NumPy':>12} | {'Dot CUDA':>12} | {'Dot CuPy':>12}")
print("-" * 95)

for N in Ns:
    # ── Python puro: só testado para N pequeno (seria lento demais acima de 100K) ──
    t_py_soma, t_py_dot = (np.nan, np.nan)  # NaN = 'não medido' (aparecerá vazio)
    if N <= 100_000:
        a_py = [1.0] * N                 # Lista Python com N uns
        b_py = [float(i) for i in range(N)]  # Lista [0.0, 1.0, 2.0, ...]
        t0 = time.perf_counter()
        _ = soma_vetores_python(a_py, b_py)
        t_py_soma = time.perf_counter() - t0  # Tempo da soma em Python puro
        t0 = time.perf_counter()
        _ = multiplicacao_vetorial_python(a_py, b_py)
        t_py_dot = time.perf_counter() - t0   # Tempo do dot em Python puro

    # ── As três versões rápidas (CPU NumPy, GPU Numba, GPU CuPy) ──
    ts_np, td_np = benchmark_numpy(N, repeticoes=repeticoes)       # CPU baseline
    ts_cuda, td_cuda = benchmark_gpu_numba(N, repeticoes=repeticoes)  # GPU manual
    ts_cupy, td_cupy = benchmark_cupy(N, repeticoes=repeticoes)       # GPU automática

    # Monta o registro (linha) com tempos em milissegundos (segundos * 1000)
    # e speedups = tempo_cpu / tempo_gpu (quanto a GPU foi mais rápida).
    registros.append({
        "N": N,
        "Python_Soma_ms": t_py_soma * 1000 if not np.isnan(t_py_soma) else np.nan,
        "Python_Dot_ms": t_py_dot * 1000 if not np.isnan(t_py_dot) else np.nan,
        "NumPy_Soma_ms": ts_np * 1000,
        "NumPy_Dot_ms": td_np * 1000,
        "CUDA_Soma_ms": ts_cuda * 1000,
        "CUDA_Dot_ms": td_cuda * 1000,
        "CuPy_Soma_ms": ts_cupy * 1000,
        "CuPy_Dot_ms": td_cupy * 1000,
        "Speedup_Soma_CUDA": ts_np / ts_cuda,  # >1 significa GPU mais rápida
        "Speedup_Soma_CuPy": ts_np / ts_cupy,
        "Speedup_Dot_CUDA": td_np / td_cuda,
        "Speedup_Dot_CuPy": td_np / td_cupy,
    })

    # Imprime a linha do N atual formatada
    print(f"{N:>12,} | {ts_np*1000:>10.3f}ms | {ts_cuda*1000:>10.3f}ms | {ts_cupy*1000:>10.3f}ms | {td_np*1000:>10.3f}ms | {td_cuda*1000:>10.3f}ms | {td_cupy*1000:>10.3f}ms")

df_res = pd.DataFrame(registros)  # Converte a lista de registros em tabela
df_res


## 6. Visualização Gráfica Interativa com Matplotlib

Geramos os gráficos comparativos de tempo absoluto e curva de speedup da GPU em relação ao NumPy.


In [ ]:
# @title 📈 Gráficos de Speedup e Tempo de Execução
# ============================================================================
# OBJETIVO: Visualizar (1) os tempos absolutos e (2) o speedup da GPU.
# ============================================================================
import matplotlib.pyplot as plt  # Biblioteca de gráficos

# Cria uma figura com 2 subgráficos lado a lado (1 linha, 2 colunas)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Benchmark de Desempenho: CPU (NumPy) vs GPU (CUDA Numba & CuPy)", fontsize=14, fontweight="bold")

# ── Gráfico 1: Tempos Absolutos de Soma Vetorial (barras) ──
ax1 = axes[0]
x_labels = [f"{n:,}" for n in df_res["N"]]  # Rótulos do eixo X (ex.: '10,000')
x = np.arange(len(x_labels))  # Posições numéricas das barras
largura = 0.25                # Largura de cada barra

# Três barras por tamanho de N, deslocadas para não se sobreporem:
ax1.bar(x - largura, df_res["NumPy_Soma_ms"], largura, label="NumPy (CPU)", color="#2563EB", alpha=0.85)
ax1.bar(x,          df_res["CUDA_Soma_ms"],  largura, label="CUDA Numba (GPU)", color="#10B981", alpha=0.85)
ax1.bar(x + largura, df_res["CuPy_Soma_ms"],  largura, label="CuPy (GPU)", color="#8B5CF6", alpha=0.85)

ax1.set_xlabel("Tamanho do Vetor (N)")
ax1.set_ylabel("Tempo (ms) — escala log")
ax1.set_title("Tempo de Execução — Soma Vetorial")
ax1.set_xticks(x)                                  # Marca as posições no eixo X
ax1.set_xticklabels(x_labels, rotation=30, ha="right")  # Rótulos inclinados p/ caber
ax1.set_yscale("log")   # Escala logarítmica: tempos variam de milissegundos a segundos
ax1.legend()            # Mostra a legenda com as cores
ax1.grid(axis="y", linestyle="--", alpha=0.4)  # Grade horizontal tracejada

# ── Gráfico 2: Speedup (NumPy / GPU) ──────────────────────
ax2 = axes[1]
speedup_cuda = df_res["Speedup_Soma_CUDA"]  # Quantas vezes a GPU Numba foi mais rápida
speedup_cupy = df_res["Speedup_Soma_CuPy"]  # Quantas vezes a GPU CuPy foi mais rápida

# Linhas de speedup para as duas abordagens de GPU
ax2.plot(x_labels, speedup_cuda, marker="o", linewidth=2, label="Speedup CUDA / NumPy", color="#10B981")
ax2.plot(x_labels, speedup_cupy, marker="s", linewidth=2, label="Speedup CuPy / NumPy", color="#8B5CF6")
# Linha de referência em 1x: abaixo dela a GPU NÃO compensa
ax2.axhline(y=1, color="gray", linestyle="--", linewidth=1.5, label="Sem ganho (1x)")

ax2.set_xlabel("Tamanho do Vetor (N)")
ax2.set_ylabel("Speedup (Tempo CPU / Tempo GPU)")
ax2.set_title("Curva de Speedup da GPU")
ax2.set_xticklabels(x_labels, rotation=30, ha="right")
ax2.legend()
ax2.grid(axis="both", linestyle="--", alpha=0.4)

# Anota o valor do speedup do CUDA sobre cada ponto da linha
for i, (sc, sp) in enumerate(zip(speedup_cuda, speedup_cupy)):
    ax2.annotate(f"{sc:.1f}x", (i, sc), textcoords="offset points", xytext=(0,6), ha='center', fontsize=8, fontweight='bold', color='#10B981')

plt.tight_layout()  # Ajusta espaçamento para evitar sobreposição
# Salva a figura em PNG com 150 dpi (boa resolução)
plt.savefig("speedup_cpu_vs_gpu_aula13.png", dpi=150, bbox_inches="tight")
plt.show()  # Exibe o gráfico no notebook
print("📸 Gráfico salvo como 'speedup_cpu_vs_gpu_aula13.png'")


## 7. Seção Extra: Requisito da Tarefa Final (`np.linalg.norm` vs `cp.linalg.norm`)

Conforme exigido na tarefa final do Bloco 2, avaliamos o cálculo de norma de vetor em CPU vs GPU.


In [ ]:
# @title 🧪 Benchmark Extra: `np.linalg.norm` vs `cp.linalg.norm`
# ============================================================================
# OBJETIVO: Comparar o cálculo da NORMA de um vetor (√soma dos quadrados)
# em CPU (NumPy) e GPU (CuPy). É o requisito da tarefa final do Bloco 2.
# ============================================================================
N_extra = 10_000_000  # Vetor de 10 milhões de elementos
a_np = np.ones(N_extra, dtype=np.float32)  # Vetor na RAM (CPU)
a_cp = cp.ones(N_extra, dtype=cp.float32)  # Vetor na VRAM (GPU)

# Warm-up de ambas as implementações fora do cronômetro
_ = np.linalg.norm(a_np)
_ = cp.linalg.norm(a_cp)
cp.cuda.runtime.deviceSynchronize()

# Benchmark CPU: 30 execuções de np.linalg.norm
t0 = time.perf_counter()
for _ in range(30):
    norma_cpu = np.linalg.norm(a_np)
t_norm_cpu = (time.perf_counter() - t0) / 30  # Tempo médio

# Benchmark GPU: 30 execuções de cp.linalg.norm (sincroniza ao final)
t0 = time.perf_counter()
for _ in range(30):
    norma_gpu = cp.linalg.norm(a_cp)
cp.cuda.runtime.deviceSynchronize()
t_norm_gpu = (time.perf_counter() - t0) / 30  # Tempo médio

# Exibe os resultados e o speedup (quantas vezes a GPU foi mais rápida)
print(f"N = {N_extra:,}")
print(f"Tempo `np.linalg.norm` (CPU): {t_norm_cpu * 1000:.3f} ms")
print(f"Tempo `cp.linalg.norm` (GPU): {t_norm_gpu * 1000:.3f} ms")
print(f"Speedup Norma Vetorial     : {t_norm_cpu / t_norm_gpu:.1f}x")


## 8. Mini-Relatório Técnico & Síntese Final do Bloco 2


In [ ]:
# @title 📑 Gerador de Mini-Relatório Técnico Automático
# ============================================================================
# OBJETIVO: Extrair conclusões automáticas do DataFrame e imprimir um relatório.
# ============================================================================

# Maior speedup do CUDA e o N onde ele ocorreu:
speedup_max = df_res["Speedup_Soma_CUDA"].max()   # Valor máximo de speedup
idx_max = df_res["Speedup_Soma_CUDA"].idxmax()     # Índice da linha desse máximo
N_max = df_res.loc[idx_max, "N"]                   # Tamanho de N correspondente

# Primeiro N em que a GPU passou a ser mais rápida que o NumPy (speedup > 1.0):
idx_compensacao = df_res[df_res["Speedup_Soma_CUDA"] > 1.0].index.min()
# Se não houver nenhum, usa 'N/A'
N_limiar = df_res.loc[idx_compensacao, "N"] if pd.notna(idx_compensacao) else "N/A"

# Imprime o relatório com os valores calculados dinamicamente
print(f"""
========================================================================
📊 MINI-RELATÓRIO TÉCNICO: DESEMPENHO PARALELO CPU VS GPU
========================================================================
1. RESUMO DOS RESULTADOS:
   - Speedup Máximo (NumPy ➔ CUDA): {speedup_max:.1f}x obtido em N = {N_max:,}.
   - Limiar de Eficiência da GPU: A GPU começou a superar o NumPy a partir de N = {N_limiar:,}.

2. ANÁLISE DE OVERHEAD DE MEMÓRIA:
   - Para tamanhos pequenos de vetor (N < 100K), a GPU apresenta desempenho inferior (speedup < 1x).
   - Isso ocorre devido ao custo fixo de lançamento de kernels (launch overhead) e ao transporte
     de dados via barramento PCIe (Host ➔ Device).

3. COMPARATIVO DAS ABORDAGENS:
   - Python Puro: Inviável para workloads de grande porte (>100K iter/s gargalo no interpretador).
   - NumPy (CPU): Excelente para protótipos e vetores pequenos/médios graças às rotinas BLAS.
   - CuPy (GPU): API idêntica ao NumPy com speedup imediato sem verbosidade de código.
   - Numba CUDA (GPU): Controle total sobre a hierarquia de threads e memória compartilhada.

4. RESPOSTA À TAREFA FINAL:
   No hardware testado, a GPU passa a compensar a partir de N ≈ {N_limiar:,}, atingindo
   sua maior eficiência em N ≥ 10M, onde a densidade computacional oculta a latência de transferência.
========================================================================
""")


---
## 📝 9. Questionário de Consolidação (Aulas 8 a 13)

As **18 questões** de revisão (Aulas 8 a 13) estão na pasta de questionários do curso:

- **Questionário:** [`questionarios/questionario-aulas-8-13.md`](../../questionarios/questionario-aulas-8-13.md)

> **Entrega:** envie as respostas por e-mail para `03049691093@senacrs.edu.br` com o assunto `Questionario aulas 8 a 13`.